# PA3 Model Prep / Runtime Prep (ONLINE, CPU only)
Run this separately for KIND="runtime" and KIND="model" in two fresh notebooks.
Internet ON, accelerator None. This installs dependencies/downloads inputs; it NEVER loads Qwen or reads InspecSafe. The inference notebook remains Internet OFF.
Use audited merged source. This notebook and its generated outputs have not been executed on Kaggle in Codex. See notes/pa3_task5_kaggle_runbook.md for exact files and remaining blockers.


In [ ]:
KIND = "runtime"  # second fresh notebook: "model"
SOURCE_COMMIT = ""  # audited merge SHA
SOURCE_BUNDLE_INPUT = ""  # input-relative path to pa3_source.bundle
SOURCE_BUNDLE_SHA256 = ""  # independently verified sha256


In [ ]:
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import subprocess
import sys

work = Path.cwd()
inputs = work.parent / "input"
if KIND not in {"runtime", "model"} or not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise ValueError("SET_KIND_AND_REVIEWED_SOURCE_SHA")
p = PurePosixPath(SOURCE_BUNDLE_INPUT)
if not SOURCE_BUNDLE_INPUT or p.is_absolute() or ".." in p.parts or "\\" in SOURCE_BUNDLE_INPUT or ":" in SOURCE_BUNDLE_INPUT:
    raise ValueError("INPUT_RELATIVE_SOURCE_BUNDLE_REQUIRED")
bundle = inputs / p
if not bundle.resolve().is_relative_to(inputs.resolve()):
    raise ValueError("BUNDLE_ESCAPES_INPUT")
def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
if sha256(bundle) != SOURCE_BUNDLE_SHA256:
    raise ValueError("SOURCE_BUNDLE_CHECKSUM")
repo = work / "data/processed" / ("pa3-prep-" + KIND)
if repo.exists():
    raise FileExistsError("FRESH_PREP_NOTEBOOK_REQUIRED")
subprocess.run(["git", "-c", "core.autocrlf=false", "clone", "--no-checkout", str(bundle), str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "-c", "core.autocrlf=false", "checkout", "--detach", SOURCE_COMMIT], check=True)
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "", "PYTHONPATH": str(repo), "PYTHONNOUSERSITE": "1"}
if KIND == "model":
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface-hub==0.30.2"], check=True, env=env)
subprocess.run([sys.executable, "-m", "safeshift.pa3.native.prep", KIND,
    "--output", "data/processed/pa3-" + KIND + "-prep", "--execute-online-owner-prep"], cwd=repo, env=env, check=True)
print("Download only the documented archive, inventory and receipts from:", repo / "data/processed" / ("pa3-" + KIND + "-prep"))
